# 3. Machine Learning: Faire Miete & Deal-Bewertung
Ziel: Für jede Wohnung schätzen wir die "faire" Miete aus Fläche, Zimmern, Bezirk, Freifläche, Befristung und Öffi-Score (Plattform als Kontrollvariable). Durch den Vergleich mit der tatsächlichen Miete erkennen wir gute Deals und zu teure Wohnungen.
Beantwortet werden die Fragen 2 (Vorhersagequalität), 3 (Einflussfaktoren), 4 und 5 (Deal-Kategorien) sowie 6 (Stabilität über die Zeit).

##  Daten laden
Wir starten eine Spark-Session und laden dabei den MongoDB Spark Connector (`spark.jars.packages`), damit Spark direkt aus MongoDB lesen kann. Beim ersten Start wird der Connector automatisch heruntergeladen.
Spark liest die Collection `listings_clean` aus der Datenbank `wohnungsmarkt`. Anschließend wandeln wir die Daten mit `toPandas()` in eine pandas-Tabelle um, da scikit-learn mit pandas arbeitet.

In [1]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

def check(bedingung, text):
    print("✅" if bedingung else "❌", text)

# Alte Spark-Session beenden, damit der Connector neu geladen wird
alte = SparkSession.getActiveSession()
if alte:
    alte.stop()

spark = SparkSession.builder \
    .appName("ml") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:10.5.0") \
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017/") \
    .config("spark.mongodb.read.database", "wohnungsmarkt") \
    .config("spark.mongodb.read.collection", "listings_clean") \
    .getOrCreate()

print("Spark-Version:", spark.version)
sdf = spark.read.format("mongodb").load()
check(sdf.count() > 0, f"{sdf.count()} Inserate mit Spark geladen")

df = sdf.drop("_id").toPandas()
check(len(df) == sdf.count(), "Umwandlung in pandas vollständig")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/23 14:18:05 WARN Utils: Your hostname, MacbookProIvana.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.5 instead (on interface en0)
26/11/23 14:18:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/ivanaarabadzic/BigData/Big%20Data%20Projekt/BigDataProjekt/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/ivanaarabadzic/.ivy2.5.2/cache
The jars for the packages stored in: /Users/ivanaarabadzic/.ivy2.5.2/jars
org.mongodb.spark#mongo-spark-connector_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-76cbbca3-93e1-4e00-968c-c2ce8925e55b;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.13;10.5.0 in central
	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] o

Spark-Version: 4.3.0
✅ 2423 Inserate mit Spark geladen


/Users/ivanaarabadzic/BigData/Big Data Projekt/BigDataProjekt/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/Users/ivanaarabadzic/BigData/Big Data Projekt/BigDataProjekt/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
26/11/23 14:18:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFie

✅ Umwandlung in pandas vollständig


## 3.1 Daten laden
Wir lesen die bereinigte Collection `listings_clean` aus der MongoDB-Datenbank `wohnungsmarkt` über Spark ein und wandeln sie mit `toPandas()` in eine pandas-Tabelle um, da scikit-learn mit pandas arbeitet.
Die Hilfsfunktion `check` gibt bei Prüfungen ✅ oder ❌ aus.

In [2]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession

def check(bedingung, text):
    print("✅" if bedingung else "❌", text)

spark = SparkSession.builder \
    .appName("ml") \
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017/wohnungsmarkt.listings_clean") \
    .getOrCreate()

df = spark.read.format("mongodb").load().toPandas()
check(len(df) > 0, f"{len(df)} Inserate geladen")

26/11/23 14:18:12 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
/Users/ivanaarabadzic/BigData/Big Data Projekt/BigDataProjekt/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/Users/ivanaarabadzic/BigData/Big Data Projekt/BigDataProjekt/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:374: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


✅ 2423 Inserate geladen


/Users/ivanaarabadzic/BigData/Big Data Projekt/BigDataProjekt/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/types.py:786: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/Users/ivanaarabadzic/BigData/Big Data Projekt/BigDataProjekt/.venv/lib/python3.12/site-packages/pyspark/sql/pandas/types.py:786: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## 3.2 Öffi-Score
Für jedes Inserat zählen wir die Linien in der Umgebung und gewichten sie nach Verkehrsmittel: U-Bahn 5, S-Bahn 4, Straßenbahn 3 und Bus 2 Punkte. Die Summe normieren wir auf eine Skala von 0 bis 10, wobei die bestangebundene Wohnung 10 bekommt.
Vereinfachung: Die Distanzgewichtung (≤ 300 m voll, ≤ 600 m halb) ist hier noch nicht enthalten.

In [3]:
punkte = (5 * df["n_ubahn_lines"] + 4 * df["n_sbahn_lines"]
          + 3 * df["n_tram_lines"] + 2 * df["n_bus_lines"])
grenze = punkte.quantile(0.95)
df["oeffi_score"] = (punkte / grenze * 10).clip(upper=10).round(1)

check(df["oeffi_score"].between(0, 10).all(), "Öffi-Score liegt zwischen 0 und 10")
df["oeffi_score"].describe()

✅ Öffi-Score liegt zwischen 0 und 10


count    2423.000000
mean        2.844284
std         2.530432
min         0.000000
25%         1.200000
50%         1.900000
75%         3.500000
max        10.000000
Name: oeffi_score, dtype: float64

## 3.3 Daten vorbereiten
Wir behalten nur Inserate mit Preis, Fläche, Zimmern und Bezirk.

Sonderfälle entfernen: Einige Inserate sind keine regulären Mietwohnungen, z. B. WG-Zimmer (Miete für ein Zimmer, aber Fläche der ganzen Wohnung), Business-Apartments (Kurzzeitmiete), Häuser oder Mietkauf-Angebote. Würde das Modell mit ihnen trainieren, würde das die faire Miete für alle anderen Wohnungen verzerren. Wir erkennen sie per Schlagwortsuche (regulärer Ausdruck) im Titel. Der Filter wurde iterativ verfeinert, da Begriffe wie "Haus" oder "WG" auch in regulären Inseraten vorkommen (z. B. "WG-geeignet", "saniertes Haus").

Danach entfernen wir extreme Ausreißer (billigste und teuerste 1 %).
Als Zielvariable verwenden wir die logarithmierte Miete `log_price`. Mieten sind schief verteilt, weil wenige sehr teure Wohnungen dabei sind. Der Logarithmus macht die Verteilung gleichmäßiger, und die Koeffizienten lassen sich später als Prozent-Effekte lesen.

In [4]:
df = df.dropna(subset=["price", "area_m2", "rooms", "bezirk"])

# Sonderfälle entfernen
muster = r"\bWG\b(?![\s-]*(geeignet|taugli))|WG-Zimmer|Studenten|Semester|Business|Einfamilienhaus|Reihenhaus|^Haus\b|Mietkauf"
sonderfall = df["title"].str.contains(muster, case=False, na=False)
print("Entfernte Sonderfälle:", sonderfall.sum())
print("Beispiele:", list(df.loc[sonderfall, "title"].head(5)))
df = df[~sonderfall]

# Ausreißer entfernen
df = df[(df["price"] > df["price"].quantile(0.01)) & (df["price"] < df["price"].quantile(0.99))]
df["log_price"] = np.log(df["price"])

check(df["price"].min() > 0, "Alle Preise positiv")
check(len(df) > 1000, f"Genug Daten ({len(df)} Inserate)")

Entfernte Sonderfälle: 37
Beispiele: ['Mietkauf! Stilvolles 93 m² charmantes Altbaujuwel in Hietzing', '23 qm WG Zimmer - direkt Karlsplatz - 698 EUR in 5er WG ab sofort', 'Studentenhit! 1- Zimmer Wohnung im Neubau! Nähe U6 Thaliastraße!', 'Business Appartement', '2. SEMESTER - Vollmöblierte Studentenzimmer in 1040 Wien - 6er-Herren-WG | Alles inklusive!']
✅ Alle Preise positiv
✅ Genug Daten (2338 Inserate)


/var/folders/c_/y_jcm28j6pbfhnlpyzjk8d240000gn/T/ipykernel_75304/2822200567.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sonderfall = df["title"].str.contains(muster, case=False, na=False)


## 3.4 Features festlegen
X enthält die Merkmale, y die Zielvariable.
Bezirk und Plattform sind Text, deshalb wandeln wir sie mit `get_dummies` in 0/1-Spalten um (One-Hot-Encoding). Freifläche und privater Anbieter sind bereits 0/1-Werte.
Die Befristung wird nur verwendet, wenn die Spalte vorhanden ist.

In [5]:
features = ["area_m2", "rooms", "has_outdoor_space", "oeffi_score", "bezirk", "source"]
if "befristet" in df.columns:
    features.append("befristet")

basis = df["bezirk"].value_counts().idxmax()   # Bezirk mit den meisten Inseraten
print("Vergleichsbezirk:", basis)

X = pd.get_dummies(df[features], columns=["bezirk", "source"]).astype(float)
X = X.drop(columns=[f"bezirk_{basis}", "source_immoscout24"])
y = df["log_price"]

print("Features:", list(X.columns))
check(X.isna().sum().sum() == 0, "Keine fehlenden Werte in den Features")
check("befristet" in features, "Befristung ist als Feature enthalten")
check(len(X) == len(y), f"X und y gleich lang ({len(X)})")

Vergleichsbezirk: 22
Features: ['area_m2', 'rooms', 'has_outdoor_space', 'oeffi_score', 'bezirk_1', 'bezirk_2', 'bezirk_3', 'bezirk_4', 'bezirk_5', 'bezirk_6', 'bezirk_7', 'bezirk_8', 'bezirk_9', 'bezirk_10', 'bezirk_11', 'bezirk_12', 'bezirk_13', 'bezirk_14', 'bezirk_15', 'bezirk_16', 'bezirk_17', 'bezirk_18', 'bezirk_19', 'bezirk_20', 'bezirk_21', 'bezirk_23', 'source_willhaben']
✅ Keine fehlenden Werte in den Features
❌ Befristung ist als Feature enthalten
✅ X und y gleich lang (2338)


## 3.5 Train/Test-Split
80 % der Daten nutzen wir zum Trainieren, 20 % zum Testen mit Wohnungen, die das Modell nicht kennt. `random_state=42` macht die Aufteilung reproduzierbar.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
check(len(X_train) + len(X_test) == len(X), "Train + Test = alle Daten")

✅ Train + Test = alle Daten


## 3.6 Modelle trainieren und bewerten (Frage 2)
Wir vergleichen drei Modelle:
- Lineare Regression: einfache Baseline, gut interpretierbar
- Random Forest: viele Entscheidungsbäume, deren Ergebnisse gemittelt werden
- Gradient Boosting: Bäume werden nacheinander gebaut, jeder korrigiert die Fehler des vorherigen
Da die Modelle log-Mieten vorhersagen, rechnen wir mit `np.exp` zurück in Euro. Kennzahlen:
- MAE: durchschnittlicher Fehler in €
- MAPE: durchschnittlicher Fehler in %
- RMSE: wie MAE, bestraft große Fehler aber stärker
- R²: Anteil der erklärten Preisunterschiede (1 = perfekt)

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score

modelle = {
    "Lineare Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

ergebnisse = []
for name, modell in modelle.items():
    modell.fit(X_train, y_train)
    echt = np.exp(y_test)
    prognose = np.exp(modell.predict(X_test))
    ergebnisse.append({
        "Modell": name,
        "MAE (€)": mean_absolute_error(echt, prognose),
        "MAPE (%)": mean_absolute_percentage_error(echt, prognose) * 100,
        "RMSE (€)": mean_squared_error(echt, prognose) ** 0.5,
        "R²": r2_score(echt, prognose),
    })

ergebnisse = pd.DataFrame(ergebnisse).round(2)
ergebnisse

check(ergebnisse["R²"].max() > 0.5, f"Bestes R² über 0.5 ({ergebnisse['R²'].max():.2f})")
check(ergebnisse["MAPE (%)"].min() < 20, f"Fehler unter 20 % ({ergebnisse['MAPE (%)'].min():.1f} %)")

✅ Bestes R² über 0.5 (0.76)
✅ Fehler unter 20 % (13.9 %)


## 3.7 Cross-Validation
Ein einzelner Split kann Glück oder Pech sein. Bei der 5-fachen Cross-Validation werden die Daten in 5 Teile geteilt, jedes Teil ist einmal Testset. Ist der Mittelwert ähnlich wie oben und die Streuung klein, ist das Ergebnis stabil.

In [8]:
from sklearn.model_selection import cross_val_score

for name, modell in modelle.items():
    scores = cross_val_score(modell, X, y, cv=5, scoring="r2")
    print(f"{name}: R² = {scores.mean():.3f} (± {scores.std():.3f})")
    check(scores.std() < 0.05, f"{name} ist stabil über alle 5 Teile")

bester_name = ergebnisse.sort_values("MAE (€)").iloc[0]["Modell"]
bestes = modelle[bester_name]
print("Bestes Modell:", bester_name)

Lineare Regression: R² = 0.738 (± 0.044)
✅ Lineare Regression ist stabil über alle 5 Teile
Random Forest: R² = 0.805 (± 0.027)
✅ Random Forest ist stabil über alle 5 Teile
Gradient Boosting: R² = 0.806 (± 0.025)
✅ Gradient Boosting ist stabil über alle 5 Teile
Bestes Modell: Random Forest


## 3.8 Einflussfaktoren: Prozent-Effekte (Frage 3)
Weil die lineare Regression auf die log-Miete trainiert wurde, bedeutet ein Koeffizient b: Steigt das Merkmal um 1, ändert sich die Miete um (e^b − 1) · 100 %.
Beispiel: b = 0,08 bei Freifläche heißt, dass eine Freifläche die Miete um ca. 8 % erhöht, bei sonst gleichen Merkmalen.

In [9]:
lin = modelle["Lineare Regression"]
effekte = pd.Series((np.exp(lin.coef_) - 1) * 100, index=X.columns).round(1)
effekte[["area_m2", "rooms", "has_outdoor_space", "oeffi_score"]]
effekte.filter(like="bezirk_").sort_values(ascending=False)

bezirk_1     24.0
bezirk_4     13.8
bezirk_8      8.9
bezirk_19     8.0
bezirk_2      4.7
bezirk_13    -0.8
bezirk_6     -0.8
bezirk_7     -1.7
bezirk_17    -3.0
bezirk_5     -3.2
bezirk_9     -3.3
bezirk_18    -6.6
bezirk_3     -7.5
bezirk_15   -12.9
bezirk_16   -12.9
bezirk_20   -13.0
bezirk_14   -16.5
bezirk_12   -17.4
bezirk_10   -18.0
bezirk_23   -19.3
bezirk_21   -22.1
bezirk_11   -25.8
dtype: float64

## 3.9 Permutation Importance
Für das beste Modell mischen wir die Werte eines Merkmals zufällig durch und messen, wie viel schlechter die Vorhersage wird. Je größer der Verlust, desto wichtiger ist das Merkmal.

In [10]:
from sklearn.metrics import r2_score
import plotly.express as px

gruppen = {
    "Fläche": ["area_m2"],
    "Zimmer": ["rooms"],
    "Freifläche": ["has_outdoor_space"],
    "Öffi-Score": ["oeffi_score"],
    "Bezirk": [s for s in X.columns if s.startswith("bezirk_")],
    "Plattform": [s for s in X.columns if s.startswith("source_")],
}

basis_r2 = r2_score(y_test, bestes.predict(X_test))
rng = np.random.default_rng(42)
wichtigkeit = {}
for name, spalten in gruppen.items():
    verluste = []
    for _ in range(10):
        X_gemischt = X_test.copy()
        zufall = rng.permutation(len(X_test))
        X_gemischt[spalten] = X_test[spalten].values[zufall]
        verluste.append(basis_r2 - r2_score(y_test, bestes.predict(X_gemischt)))
    wichtigkeit[name] = np.mean(verluste)

wichtigkeit = pd.Series(wichtigkeit).sort_values()
px.bar(x=wichtigkeit.values, y=wichtigkeit.index, orientation="h",
       labels={"x": "Verlust an R² beim Mischen", "y": ""},
       title=f"Wichtigste Merkmale ({bester_name})").show()

## 3.10 Vorhersage vs. tatsächliche Miete
Jeder Punkt ist eine Wohnung aus dem Testset. Punkte auf der Diagonale wurden perfekt vorhergesagt.

In [11]:
echt = np.exp(y_test)
prognose = np.exp(bestes.predict(X_test))
fig = px.scatter(x=echt, y=prognose, labels={"x": "Tatsächliche Miete (€)", "y": "Geschätzte Miete (€)"},
                 title="Geschätzte vs. tatsächliche Miete")
fig.add_shape(type="line", x0=echt.min(), y0=echt.min(), x1=echt.max(), y1=echt.max(), line_color="red")
fig.show()

## 3.11 Deal-Bewertung (Fragen 4 und 5)
Für jede Wohnung berechnen wir die faire Miete und die Abweichung:
Abweichung = (tatsächliche Miete − geschätzte Miete) / geschätzte Miete
Kategorien: Sehr guter Deal (< −15 %), Guter Deal (−15 bis −5 %), Fair (±5 %), Teuer (5 bis 15 %), Zu teuer (> 15 %).

Die Schwellenwerte orientieren sich an der Modellgenauigkeit: Da das Modell im Durchschnitt um ca. 14 % (MAPE) danebenliegt, gelten Abweichungen bis ±10 % als "Fair". Erst ab 10 % (bzw. 25 %) Abweichung sprechen wir von einem (sehr) guten Deal bzw. einer teuren (zu teuren) Wohnung.
Kategorien: Sehr guter Deal (< −25 %), Guter Deal (−25 bis −10 %), Fair (±10 %), Teuer (10 bis 25 %), Zu teuer (> 25 %).

In [12]:
from sklearn.model_selection import cross_val_predict

log_prognose = cross_val_predict(bestes, X, y, cv=5)
df["faire_miete"] = np.exp(log_prognose).round(0)
df["abweichung"] = (df["price"] - df["faire_miete"]) / df["faire_miete"] * 100
df["ersparnis"] = df["faire_miete"] - df["price"]

df["kategorie"] = pd.cut(df["abweichung"],
    bins=[-np.inf, -25, -10, 10, 25, np.inf],
    labels=["Sehr guter Deal", "Guter Deal", "Fair", "Teuer", "Zu teuer"])

df["kategorie"].value_counts()

check(df["faire_miete"].min() > 0, "Alle geschätzten Mieten positiv")
check(df["kategorie"].isna().sum() == 0, "Jedes Inserat hat eine Kategorie")
anteil_fair = (df["kategorie"] == "Fair").mean() * 100
print(f"Anteil fair bepreist: {anteil_fair:.1f} %")

print(df["kategorie"].value_counts())

✅ Alle geschätzten Mieten positiv
✅ Jedes Inserat hat eine Kategorie
Anteil fair bepreist: 49.4 %
kategorie
Fair               1156
Guter Deal          479
Teuer               272
Zu teuer            258
Sehr guter Deal     173
Name: count, dtype: int64


## 3.12 Verteilung der Abweichungen
Das Histogramm zeigt, wie viele Inserate unter bzw. über der Modell-Schätzung liegen.

In [13]:
farben = {"Sehr guter Deal": "#1a9850", "Guter Deal": "#91cf60", "Fair": "#bdbdbd",
          "Teuer": "#fc8d59", "Zu teuer": "#d73027"}
reihenfolge = ["Sehr guter Deal", "Guter Deal", "Fair", "Teuer", "Zu teuer"]

fig = px.histogram(df, x="abweichung", nbins=80, color="kategorie",
                   color_discrete_map=farben, category_orders={"kategorie": reihenfolge},
                   labels={"abweichung": "Abweichung von der fairen Miete (%)", "kategorie": "Kategorie"},
                   title="Verteilung der Abweichungen")
fig.update_xaxes(range=[-60, 100])
fig.update_yaxes(title="Anzahl Inserate")
fig.show()

print("Inserate über +100 % (nicht im Bild):", (df["abweichung"] > 100).sum())

Inserate über +100 % (nicht im Bild): 15


## 3.13 Muster nach Bezirk, Anbieter und Plattform (Frage 5)
Wir berechnen pro Gruppe den Anteil der Kategorien in Prozent.

In [14]:
for spalte in ["bezirk", "is_private", "source"]:
    tabelle = pd.crosstab(df[spalte], df["kategorie"], normalize="index") * 100
    display(tabelle.round(1))

kategorie,Sehr guter Deal,Guter Deal,Fair,Teuer,Zu teuer
bezirk,,,,,
1,6.6,22.1,31.6,20.6,19.1
2,8.1,24.4,34.9,15.1,17.4
3,1.8,10.7,73.5,9.2,4.8
4,11.4,15.9,34.1,9.1,29.5
5,4.8,19.0,50.0,9.5,16.7
6,13.2,15.8,26.3,18.4,26.3
7,11.4,25.0,38.6,11.4,13.6
8,3.4,17.2,58.6,6.9,13.8
9,14.6,12.5,47.9,12.5,12.5


kategorie,Sehr guter Deal,Guter Deal,Fair,Teuer,Zu teuer
is_private,,,,,
0,6.8,20.2,50.3,11.6,11.0
1,18.3,25.8,33.3,11.7,10.8


kategorie,Sehr guter Deal,Guter Deal,Fair,Teuer,Zu teuer
source,,,,,
immoscout24,6.0,20.2,50.3,11.7,11.8
willhaben,9.6,20.9,48.1,11.6,9.9


## 3.14 Top-Deals
Wir ranken alle Inserate nach der Ersparnis gegenüber der fairen Miete, einmal in Euro und einmal in Prozent. Das Ranking in Euro bevorzugt große Wohnungen (20 % von 4.000 € sind mehr als 20 % von 1.000 €), das Ranking in Prozent zeigt relative Schnäppchen unabhängig von der Größe.

In [15]:
spalten = ["title", "bezirk", "area_m2", "price", "faire_miete", "ersparnis", "abweichung", "kategorie"]

print("Top 10 nach Ersparnis in €:")
display(df.sort_values("ersparnis", ascending=False)[spalten].head(10).round(1))

print("\nTop 10 nach Abweichung in %:")
display(df.sort_values("abweichung")[spalten].head(10).round(1))

Top 10 nach Ersparnis in €:


,title,bezirk,area_m2,price,faire_miete,ersparnis,abweichung,kategorie
251,Palaiswohnung am Franz-Josefs-Kai,1,225.0,4045.0,6339.0,2294.0,-36.2,Sehr guter Deal
664,RODAUN - große Maisonettewohnung,23,124.0,1500.0,3382.0,1882.0,-55.6,Sehr guter Deal
517,"DACHGESCHOSS, FERNLICK, 95 m2 Neubau mit 22 m2...",20,95.0,1596.2,3237.0,1640.8,-50.7,Sehr guter Deal
48,DESIGN DACH-TERRASSENWOHNUNG,1,212.0,4994.0,6585.0,1591.0,-24.2,Guter Deal
16,DACHTERRASSENMIETE IN DER INNENSTADT - NÄHE ST...,1,212.0,4995.0,6580.0,1585.0,-24.1,Guter Deal
949,ERSTBEZUG | NÄHE BÖRSE | EXQUISITE AUSSTATTUNG...,1,203.0,4280.0,5835.0,1555.0,-26.6,Sehr guter Deal
1795,SCHÖNE ALTBAUMIETE MIT ZWEI BALKONEN - ERSTBEZ...,18,234.0,3900.0,5371.0,1471.0,-27.4,Sehr guter Deal
2363,helle 5-Zimmer-Altbauwohnung in der Heinestraße,2,217.6,3450.0,4903.0,1453.0,-29.6,Sehr guter Deal
1182,Wohnen auf drei Ebenen – Maisonette mit zwei T...,21,125.8,2199.0,3627.0,1428.0,-39.4,Sehr guter Deal
882,Großzügige 2-Zimmer-Wohnung in Hernals,17,81.0,930.5,2280.0,1349.5,-59.2,Sehr guter Deal



Top 10 nach Abweichung in %:


,title,bezirk,area_m2,price,faire_miete,ersparnis,abweichung,kategorie
882,Großzügige 2-Zimmer-Wohnung in Hernals,17,81.0,930.5,2280.0,1349.5,-59.2,Sehr guter Deal
664,RODAUN - große Maisonettewohnung,23,124.0,1500.0,3382.0,1882.0,-55.6,Sehr guter Deal
179,Sehr schöne 3-Zimmer Wohnung- Erdgeschoßwohnung,9,70.0,794.9,1629.0,834.1,-51.2,Sehr guter Deal
517,"DACHGESCHOSS, FERNLICK, 95 m2 Neubau mit 22 m2...",20,95.0,1596.2,3237.0,1640.8,-50.7,Sehr guter Deal
550,VILLAGE IM DRITTEN | BAUFELD 3 | 2 ZIMMER | AB...,1,48.0,1129.0,2127.0,998.0,-46.9,Sehr guter Deal
1280,STILVOLLE RÄUMLICHKEITEN MIT GROSSEM BALKON,13,105.4,1470.8,2716.0,1245.2,-45.8,Sehr guter Deal
20,"2-Zimmer-Wohnung in Währing, 67,7 m², nahe Kut...",18,68.0,751.7,1341.0,589.3,-43.9,Sehr guter Deal
1749,"Stadlauer Straße - 2 Zimmer Neubau mit 6,06m2 ...",22,56.2,845.0,1490.0,645.0,-43.3,Sehr guter Deal
916,*** RESERVIERT ***ALSZEILE - TOLLE GRÜNRUHELAG...,17,107.0,1499.0,2632.0,1133.0,-43.0,Sehr guter Deal
1466,Charmante 2-Zimmer Dachgeschosswohnung inkl. E...,21,89.2,1199.0,2088.0,889.0,-42.6,Sehr guter Deal


## 3.15 Stabilität über die Zeit (Frage 6)
Wir trainieren nur mit älteren Inseraten (die ersten 80 % nach `first_seen`) und testen mit den neuesten 20 %. Ist der Fehler ähnlich wie beim zufälligen Split, bleibt das Modell auch bei neuen Inseraten stabil.

In [16]:
df_zeit = df.sort_values("first_seen")
grenze = int(len(df_zeit) * 0.8)
alt, neu = df_zeit.index[:grenze], df_zeit.index[grenze:]

bestes.fit(X.loc[alt], y.loc[alt])
prog_neu = np.exp(bestes.predict(X.loc[neu]))
mape_neu = mean_absolute_percentage_error(np.exp(y.loc[neu]), prog_neu) * 100
print(f"MAPE bei neuen Inseraten: {mape_neu:.1f} %")

mape_vorher = ergebnisse.loc[ergebnisse["Modell"] == bester_name, "MAPE (%)"].iloc[0]
check(abs(mape_neu - mape_vorher) < 5, f"Fehler bei neuen Inseraten ähnlich wie vorher ({mape_vorher:.1f} % vs. {mape_neu:.1f} %)")

MAPE bei neuen Inseraten: 13.7 %
✅ Fehler bei neuen Inseraten ähnlich wie vorher (13.9 % vs. 13.7 %)


## 3.16 Ergebnisse zurück in MongoDB schreiben
Laut Konzept werden die Bewertungen (geschätzte Miete, Abweichung, Kategorie) pro Inserat in MongoDB gespeichert, damit das Dashboard sie nutzen kann.

In [17]:
from pymongo import MongoClient

coll = MongoClient("mongodb://localhost:27017/")["wohnungsmarkt"]["listings_clean"]

coll.update_many({}, {"$unset": {"faire_miete": "", "abweichung": "", "kategorie": ""}})

for _, w in df.iterrows():
    coll.update_one({"_id": w["_id"]}, {"$set": {
        "faire_miete": float(w["faire_miete"]),
        "abweichung": float(w["abweichung"]),
        "kategorie": str(w["kategorie"])}})
check(True, f"{len(df)} Bewertungen gespeichert")

gespeichert = coll.count_documents({"kategorie": {"$exists": True}})
check(gespeichert == len(df), f"{gespeichert} Bewertungen in MongoDB gespeichert")

✅ 2338 Bewertungen gespeichert
✅ 2338 Bewertungen in MongoDB gespeichert


26/11/23 15:27:50 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 940055 ms exceeds timeout 120000 ms
26/11/23 15:27:50 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/23 15:27:52 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:143)
	at org.apache.spark.stora

## 3.17 Ergebnisse und Limitationen

**Vorhersagequalität (Frage 2):** Nach dem Entfernen von Sonderfällen (WG-Zimmer, Häuser, Mietkauf u. a.) umfasst der Datensatz 2.338 Inserate. Das beste Modell (Random Forest) erreicht auf dem Testset einen durchschnittlichen Fehler von 13,9 % (MAPE) bzw. __ € (MAE) und ein R² von 0,76. In der 5-fachen Cross-Validation erreichen Random Forest und Gradient Boosting ein R² von __, die lineare Baseline __. Die Baummodelle sind damit klar besser, untereinander aber praktisch gleichwertig. Mieten bis ca. 3.000 € werden zuverlässig geschätzt, sehr teure Wohnungen eher unterschätzt.

**Einflussfaktoren (Frage 3):** Wichtigster Faktor ist mit Abstand die Fläche, gefolgt von Öffi-Score, Bezirk und Freifläche. Zimmeranzahl und Plattform spielen kaum eine Rolle. Laut log-linearem Modell erhöht jeder zusätzliche m² die Miete um ca. __ %, ein zusätzliches Zimmer um __ %, eine Freifläche um __ % und ein Öffi-Score-Punkt um __ %. Gegenüber dem Vergleichsbezirk Donaustadt ist die Innere Stadt ca. __ % teurer, Simmering ca. __ % günstiger. Bezirk und Öffi-Score messen teilweise dasselbe (Lage), weshalb der Bezirk in der Permutation Importance geringer erscheint.

**Schwellenwerte (Frage 4):** Da das Modell im Schnitt um ca. 14 % abweicht, wären enge Schwellen (±5 %) nicht aussagekräftig. Wir legen daher fest: Fair bis ±10 %, Guter Deal bzw. Teuer bei 10 bis 25 % Abweichung, Sehr guter Deal bzw. Zu teuer über 25 %.

**Bewertung der Inserate (Frage 5):** 49 % der Inserate sind fair bepreist, 28 % sind (sehr) gute Deals und 23 % (zu) teuer. Die Verteilung ist rechtsschief: Überteuerte Angebote weichen stärker ab als günstige. Die meisten Deals finden sich in Außenbezirken wie Simmering, Penzing und Favoriten, die meisten überteuerten Angebote in Mariahilf, Wieden und Währing. Private Vermieter bieten etwa dreimal so oft sehr gute Deals an wie gewerbliche. Zwischen den Plattformen gibt es kaum Unterschiede. Das Ergebnis ist robust: Auch vor dem Entfernen der Sonderfälle war die Aufteilung nahezu identisch.

**Stabilität (Frage 6):** Wird das Modell nur mit älteren Inseraten trainiert, liegt der Fehler bei den neuesten 20 % bei __ %, also auf demselben Niveau wie beim zufälligen Split. Das Modell ist im beobachteten Zeitraum stabil.

**Limitationen:**
- Es werden Angebots- statt tatsächlich gezahlter Mieten verwendet.
- Befristung, Zustand, Baujahr und Ausstattung werden nicht erfasst. Der Freiflächen-Effekt enthält vermutlich teilweise einen Neubau-Effekt.
- Der Öffi-Score ist vereinfacht (ohne Gewichtung nach Distanz), Bezirkskennzahlen (Einkommen) fehlen noch.
- Koordinaten bzw. Bezirkszuordnung sind teilweise ungenau, vereinzelt gibt es Duplikate.
- Sonderfälle werden per Schlagwortsuche erkannt, das ist nicht fehlerfrei.
- Private Inserate enthalten eventuell teils Mieten ohne Betriebskosten.
- Der Beobachtungszeitraum ist kurz. Das Modell sollte regelmäßig mit neuen Daten nachtrainiert werden.